# 🎬 MSBA 305 — Movie Success Intelligence System
## Person 1: Data Acquisition + Source Appraisal

**Team:** Maria El Khoury · Amani Al Ghali · Gaby Kassab · Charbel Hajj · Antonio Chakhtoura  
**Course:** MSBA 305 — Spring 2025/2026  

---
### 📋 What this notebook does:
- **Step 1:** Fetch movies from TMDB API → saves `tmdb_raw.json`
- **Step 2:** Scrape Box Office Mojo → saves `boxoffice_raw.csv`
- **Step 3:** Load & inspect the CSV dataset
- **Step 4:** Generate Data Source Appraisal Table
---

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from google.colab import files
uploaded = files.upload()

Saving Top_200_Movies_Dataset_2023(Cleaned).csv to Top_200_Movies_Dataset_2023(Cleaned).csv


## ⚙️ Install Dependencies

In [4]:
!pip install requests beautifulsoup4 pandas tabulate --quiet

---
## 🔹 STEP 1 — TMDB API
Fetch movies from The Movie Database using the `/discover/movie` endpoint.

**Filters applied:**
- Sorted by popularity (descending)
- Vote count ≥ 500 (ensures meaningful ratings)
- Release date ≤ 2024-12-31
- No adult content

In [5]:
import requests
import json
import time

# ─── CONFIGURATION ───────────────────────────────────────────────
API_KEY  = "03ce4a961888ce66fa486f41027cf1db"   # ← your TMDB API key
BASE_URL = "https://api.themoviedb.org/3/discover/movie"
OUTPUT_FILE = "tmdb_raw.json"
TARGET_PAGES = 25   # 25 pages × 20 movies = up to 500 movies
# ─────────────────────────────────────────────────────────────────

movies   = []
seen_ids = set()
errors   = []

print(f"📡 Fetching movies from TMDB API...\n")

for page in range(1, TARGET_PAGES + 1):
    params = {
        "api_key"                    : API_KEY,
        "page"                       : page,
        "sort_by"                    : "popularity.desc",
        "primary_release_date.lte"   : "2024-12-31",
        "vote_count.gte"             : 500,
        "include_adult"              : "false",
        "include_video"              : "false"
    }

    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        if "results" not in data:
            print(f"  ⚠️ No results on page {page}: {data}")
            errors.append(page)
            break

        added = 0
        for movie in data["results"]:
            if movie["id"] not in seen_ids:
                seen_ids.add(movie["id"])
                movies.append(movie)
                added += 1

        print(f"  ✅ Page {page:02d} — Added {added} movies | Total so far: {len(movies)}")

    except requests.exceptions.RequestException as e:
        print(f"  ❌ Page {page} failed: {e}")
        errors.append(page)

    time.sleep(0.25)   # be polite to the API (rate limit)

# ── Save output ──────────────────────────────────────────────────
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(movies, f, indent=2, ensure_ascii=False)

print(f"\n{'='*50}")
print(f"✅ TMDB fetch complete!")
print(f"   Total movies collected : {len(movies)}")
print(f"   Pages with errors      : {errors if errors else 'None'}")
print(f"   Saved to               : {OUTPUT_FILE}")
print(f"{'='*50}")

📡 Fetching movies from TMDB API...

  ✅ Page 01 — Added 20 movies | Total so far: 20
  ✅ Page 02 — Added 20 movies | Total so far: 40
  ✅ Page 03 — Added 20 movies | Total so far: 60
  ✅ Page 04 — Added 20 movies | Total so far: 80
  ✅ Page 05 — Added 20 movies | Total so far: 100
  ✅ Page 06 — Added 20 movies | Total so far: 120
  ✅ Page 07 — Added 20 movies | Total so far: 140
  ✅ Page 08 — Added 20 movies | Total so far: 160
  ✅ Page 09 — Added 20 movies | Total so far: 180
  ✅ Page 10 — Added 20 movies | Total so far: 200
  ✅ Page 11 — Added 20 movies | Total so far: 220
  ✅ Page 12 — Added 20 movies | Total so far: 240
  ✅ Page 13 — Added 20 movies | Total so far: 260
  ✅ Page 14 — Added 20 movies | Total so far: 280
  ✅ Page 15 — Added 20 movies | Total so far: 300
  ✅ Page 16 — Added 20 movies | Total so far: 320
  ✅ Page 17 — Added 20 movies | Total so far: 340
  ✅ Page 18 — Added 20 movies | Total so far: 360
  ✅ Page 19 — Added 20 movies | Total so far: 380
  ✅ Page 20 — Adde

In [6]:
# 🔍 Preview first movie record to verify structure
print("\n📄 Sample TMDB record (first movie):")
print(json.dumps(movies[0], indent=2))


📄 Sample TMDB record (first movie):
{
  "adult": false,
  "backdrop_path": "/9n2tJBplPbgR2ca05hS5CKXwP2c.jpg",
  "genre_ids": [
    10751,
    35,
    12,
    14,
    16
  ],
  "id": 502356,
  "original_language": "en",
  "original_title": "The Super Mario Bros. Movie",
  "overview": "While working underground to fix a water main, Brooklyn plumbers\u2014and brothers\u2014Mario and Luigi are transported down a mysterious pipe and wander into a magical new world. But when the brothers are separated, Mario embarks on an epic quest to find Luigi.",
  "popularity": 380.5992,
  "poster_path": "/qNBAXBIQlnOThrVvA6mA2B5ggV6.jpg",
  "release_date": "2023-04-05",
  "title": "The Super Mario Bros. Movie",
  "video": false,
  "vote_average": 7.588,
  "vote_count": 10473
}


---
## 🔹 STEP 2 — Box Office Mojo (Web Scraping)
Scrape the top lifetime grossing movies from Box Office Mojo.

**Extracting:** title, lifetime gross revenue, year

In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# ─── CONFIGURATION ───────────────────────────────────────────────
BOX_OFFICE_URL = "https://www.boxofficemojo.com/chart/top_lifetime_gross/"
OUTPUT_FILE_BO = "boxoffice_raw.csv"
PAGES_TO_SCRAPE = 5   # each page has ~200 rows; 5 pages = ~1000 movies
# ─────────────────────────────────────────────────────────────────

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

all_rows = []

print("🌐 Scraping Box Office Mojo...\n")

for page_num in range(1, PAGES_TO_SCRAPE + 1):
    url = BOX_OFFICE_URL if page_num == 1 else f"{BOX_OFFICE_URL}?offset={(page_num-1)*200}"

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        table = soup.find("table")

        if not table:
            print(f"  ⚠️ No table found on page {page_num}. Box Office Mojo may have changed layout.")
            break

        rows = table.find_all("tr")
        page_rows = 0

        for row in rows[1:]:   # skip header row
            cols = row.find_all("td")
            if len(cols) < 3:
                continue

            rank     = cols[0].get_text(strip=True)
            title    = cols[1].get_text(strip=True)
            revenue  = cols[2].get_text(strip=True)

            # Try to get year from the 4th column if available
            year = cols[3].get_text(strip=True) if len(cols) > 3 else ""

            all_rows.append({
                "rank"    : rank,
                "title"   : title,
                "revenue" : revenue,
                "year"    : year
            })
            page_rows += 1

        print(f"  ✅ Page {page_num} — Scraped {page_rows} rows | Total so far: {len(all_rows)}")

    except requests.exceptions.RequestException as e:
        print(f"  ❌ Page {page_num} failed: {e}")

    time.sleep(1.0)   # be polite to the server

# ── Save output ──────────────────────────────────────────────────
df_boxoffice = pd.DataFrame(all_rows)
df_boxoffice.to_csv(OUTPUT_FILE_BO, index=False, encoding="utf-8")

print(f"\n{'='*50}")
print(f"✅ Box Office Mojo scrape complete!")
print(f"   Total rows scraped : {len(all_rows)}")
print(f"   Saved to           : {OUTPUT_FILE_BO}")
print(f"{'='*50}")

df_boxoffice.head(10)

🌐 Scraping Box Office Mojo...

  ✅ Page 1 — Scraped 200 rows | Total so far: 200
  ✅ Page 2 — Scraped 200 rows | Total so far: 400
  ✅ Page 3 — Scraped 200 rows | Total so far: 600
  ✅ Page 4 — Scraped 200 rows | Total so far: 800
  ✅ Page 5 — Scraped 200 rows | Total so far: 1000

✅ Box Office Mojo scrape complete!
   Total rows scraped : 1000
   Saved to           : boxoffice_raw.csv


,rank,title,revenue,year
0,1,Star Wars: Episode VII - The Force Awakens,"$936,662,225",2015
1,2,Avengers: Endgame,"$858,373,000",2019
2,3,Spider-Man: No Way Home,"$814,866,759",2021
3,4,Avatar,"$785,221,649",2009
4,5,Top Gun: Maverick,"$718,732,821",2022
5,6,Black Panther,"$700,426,566",2018
6,7,Avatar: The Way of Water,"$688,459,501",2022
7,8,Avengers: Infinity War,"$678,815,482",2018
8,9,Titanic,"$674,354,882",1997
9,10,Jurassic World,"$653,406,625",2015


---
## 🔹 STEP 3 — CSV Dataset (Top 200 Movies)
Load and inspect the Kaggle/reference CSV file.

In [8]:
# ── If running in Google Colab, upload the CSV first ─────────────
import os

CSV_FILENAME = "Top_200_Movies_Dataset_2023_Cleaned_.csv"

# Check if file already exists (e.g. uploaded manually or cloned from Drive)
if not os.path.exists(CSV_FILENAME):
    print("📂 File not found locally. Uploading from your computer...")
    from google.colab import files
    uploaded = files.upload()   # a file picker dialog will appear
    CSV_FILENAME = list(uploaded.keys())[0]
    print(f"Uploaded: {CSV_FILENAME}")
else:
    print(f"✅ Found file: {CSV_FILENAME}")

📂 File not found locally. Uploading from your computer...


Saving Top_200_Movies_Dataset_2023(Cleaned).csv to Top_200_Movies_Dataset_2023(Cleaned) (1).csv
Uploaded: Top_200_Movies_Dataset_2023(Cleaned) (1).csv


In [9]:
import pandas as pd

df_csv = pd.read_csv(CSV_FILENAME, encoding="utf-8-sig")

print("\n📊 CSV Dataset — Basic Info")
print(f"   Rows    : {len(df_csv)}")
print(f"   Columns : {list(df_csv.columns)}")

print("\n📋 First 5 rows:")
display(df_csv.head())

print("\n🔎 Missing values per column:")
print(df_csv.isnull().sum())

print("\n📈 Column data types:")
print(df_csv.dtypes)

print("\n✅ Useful fields identified:")
print("   • Title        → movie name (for matching)")
print("   • Total Gross  → box office revenue (USD)")
print("   • Release Date → will extract year during cleaning")
print("   • Distributor  → studio / distributor name")


📊 CSV Dataset — Basic Info
   Rows    : 200
   Columns : ['Rank', 'Title', 'Theaters', 'Total Gross', 'Release Date', 'Distributor']

📋 First 5 rows:


,Rank,Title,Theaters,Total Gross,Release Date,Distributor
0,1,Barbie,"4,337","$594,254,460",2023-07-21 00:00:00,Warner Bros.
1,2,The Super Mario Bros. Movie,"4,371","$574,759,600",2023-04-05 00:00:00,Universal Pictures
2,3,Spider-Man: Across the Spider-Verse,"4,332","$381,178,195",2023-06-02 00:00:00,Columbia Pictures
3,4,Guardians of the Galaxy Vol. 3,"4,450","$358,995,815",2023-05-05 00:00:00,Walt Disney Studios Motion Pictures
4,5,Oppenheimer,"3,761","$300,144,670",2023-07-21 00:00:00,Universal Pictures



🔎 Missing values per column:
Rank            0
Title           0
Theaters        0
Total Gross     0
Release Date    0
Distributor     0
dtype: int64

📈 Column data types:
Rank             int64
Title           object
Theaters        object
Total Gross     object
Release Date    object
Distributor     object
dtype: object

✅ Useful fields identified:
   • Title        → movie name (for matching)
   • Total Gross  → box office revenue (USD)
   • Release Date → will extract year during cleaning
   • Distributor  → studio / distributor name


In [13]:
# ⚠️ NOTE: Missing fields in CSV and where to get them
print("=== FIELD AVAILABILITY ACROSS ALL SOURCES ===")
print()
print("✅ title       → CSV + Box Office Mojo + TMDB")
print("✅ year        → CSV (extract from Release Date) + Box Office Mojo + TMDB")
print("✅ genre       → ❌ NOT in CSV → will come from TMDB (genre_ids)")
print("✅ rating      → ❌ NOT in CSV → will come from TMDB (vote_average)")
print("✅ revenue     → CSV (Total Gross) + Box Office Mojo")
print("✅ popularity  → TMDB only")
print()
print("📌 CONCLUSION: CSV alone is not enough.")
print("   TMDB fills the missing genre and rating fields.")
print("   All 3 sources together give us the complete picture.")

=== FIELD AVAILABILITY ACROSS ALL SOURCES ===

✅ title       → CSV + Box Office Mojo + TMDB
✅ year        → CSV (extract from Release Date) + Box Office Mojo + TMDB
✅ genre       → ❌ NOT in CSV → will come from TMDB (genre_ids)
✅ rating      → ❌ NOT in CSV → will come from TMDB (vote_average)
✅ revenue     → CSV (Total Gross) + Box Office Mojo
✅ popularity  → TMDB only

📌 CONCLUSION: CSV alone is not enough.
   TMDB fills the missing genre and rating fields.
   All 3 sources together give us the complete picture.


---
## 🔹 STEP 4 — Data Source Appraisal ⚠️ VERY IMPORTANT
For each source: format, volume, quality assessment, and known risks.

In [10]:
import pandas as pd
import json

# ── Collect stats from each source ───────────────────────────────

# TMDB
with open("tmdb_raw.json", encoding="utf-8") as f:
    tmdb_data = json.load(f)
tmdb_count    = len(tmdb_data)
tmdb_missing  = sum(1 for m in tmdb_data if not m.get("title") or m.get("vote_average") is None)

# Box Office Mojo
df_bo   = pd.read_csv("boxoffice_raw.csv")
bo_count    = len(df_bo)
bo_missing  = df_bo.isnull().sum().sum()

# CSV
df_csv  = pd.read_csv(CSV_FILENAME, encoding="utf-8-sig")
csv_count   = len(df_csv)
csv_missing = df_csv.isnull().sum().sum()

# ── Build appraisal table ─────────────────────────────────────────
appraisal = [
    {
        "Source"         : "TMDB API",
        "Format"         : "JSON (REST API)",
        "Volume"         : f"{tmdb_count} records",
        "Key Fields"     : "title, vote_average, popularity, genre_ids, release_date",
        "Quality"        : "High — structured, well-maintained, official source",
        "Known Risks"    : (
            "• Popularity bias: filters favor mainstream films\n"
            "• vote_count ≥ 500 filter may exclude niche/indie films\n"
            "• No direct box office revenue field\n"
            f"• Missing or null fields: ~{tmdb_missing} records"
        )
    },
    {
        "Source"         : "Box Office Mojo",
        "Format"         : "HTML (Web Scraping)",
        "Volume"         : f"{bo_count} records",
        "Key Fields"     : "title, lifetime gross revenue, year",
        "Quality"        : "Medium — scraped HTML; layout may change over time",
        "Known Risks"    : (
            "• Revenue in string format (e.g. '$1,234,567') — needs parsing\n"
            "• Website layout may change, breaking the scraper\n"
            "• Only covers North American box office (not global)\n"
            f"• Missing values: {bo_missing} cells across all columns"
        )
    },
    {
        "Source"         : "Top 200 Movies CSV",
        "Format"         : "CSV (Flat File)",
        "Volume"         : f"{csv_count} records",
        "Key Fields"     : "Title, Total Gross, Release Date, Distributor",
        "Quality"        : "High — clean, pre-processed dataset; no missing values",
        "Known Risks"    : (
            "• Limited to only 200 movies (2023 only)\n"
            "• Revenue stored as string with $ and commas — needs parsing\n"
            "• No audience rating or genre columns\n"
            f"• Missing values: {csv_missing} cells across all columns"
        )
    }
]

df_appraisal = pd.DataFrame(appraisal)

print("\n📊 DATA SOURCE APPRAISAL TABLE")
print("="*80)
display(df_appraisal)

# Save to CSV for the report
df_appraisal.to_csv("data_source_appraisal.csv", index=False, encoding="utf-8")
print("\n✅ Saved appraisal to: data_source_appraisal.csv")


📊 DATA SOURCE APPRAISAL TABLE


,Source,Format,Volume,Key Fields,Quality,Known Risks
0,TMDB API,JSON (REST API),500 records,"title, vote_average, popularity, genre_ids, re...","High — structured, well-maintained, official s...",• Popularity bias: filters favor mainstream fi...
1,Box Office Mojo,HTML (Web Scraping),1000 records,"title, lifetime gross revenue, year",Medium — scraped HTML; layout may change over ...,"• Revenue in string format (e.g. '$1,234,567')..."
2,Top 200 Movies CSV,CSV (Flat File),200 records,"Title, Total Gross, Release Date, Distributor","High — clean, pre-processed dataset; no missin...",• Limited to only 200 movies (2023 only)\n• Re...



✅ Saved appraisal to: data_source_appraisal.csv


---
## 📦 FINAL DELIVERABLES CHECKLIST

In [11]:
import os

deliverables = {
    "tmdb_raw.json"              : "✅ Raw TMDB data (JSON)",
    "boxoffice_raw.csv"          : "✅ Raw Box Office Mojo data (CSV)",
    CSV_FILENAME                 : "✅ CSV dataset loaded and inspected",
    "data_source_appraisal.csv"  : "✅ Data Source Appraisal Table"
}

print("\n📦 PERSON 1 — DELIVERABLES STATUS")
print("="*50)
all_done = True
for filename, label in deliverables.items():
    exists = os.path.exists(filename)
    status = "✅" if exists else "❌ MISSING"
    size   = f"({os.path.getsize(filename):,} bytes)" if exists else ""
    print(f"  {status}  {label} {size}")
    if not exists:
        all_done = False

print("="*50)
if all_done:
    print("\n🎉 All deliverables ready! Hand off to Person 2 (Data Cleaning).")
else:
    print("\n⚠️ Some files are missing. Re-run the relevant steps above.")


📦 PERSON 1 — DELIVERABLES STATUS
  ✅  ✅ Raw TMDB data (JSON) (378,517 bytes)
  ✅  ✅ Raw Box Office Mojo data (CSV) (41,758 bytes)
  ✅  ✅ CSV dataset loaded and inspected (15,809 bytes)
  ✅  ✅ Data Source Appraisal Table (1,110 bytes)

🎉 All deliverables ready! Hand off to Person 2 (Data Cleaning).


In [12]:
from google.colab import files
files.download('tmdb_raw.json')
files.download('boxoffice_raw.csv')
files.download('data_source_appraisal.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 🤝 Handoff to Person 2 (Data Cleaning)

Person 2 needs these files from you:

| File | Description |
|------|-------------|
| `tmdb_raw.json` | ~500 movies from TMDB API |
| `boxoffice_raw.csv` | Scraped box office revenue + titles |
| `Top_200_Movies_Dataset_2023_Cleaned_.csv` | Reference CSV |
| `data_source_appraisal.csv` | Your appraisal table |

> 💡 Share via Google Drive folder or upload to your shared Colab environment.